# 13 — Filter the relevant final Commission manifest

This notebook performs the **relevance/deletion step** after merging.

It reads the merged Commission-decision manifest created by notebook 12 and removes rows marked through `candidate_to_delete`.

A specific safeguard is applied for **negative clearances**: negative-clearance decisions are retained even if `candidate_to_delete` is populated.

All other delete behavior remains the same as before:

- blank `candidate_to_delete` → keep;
- `false`, `no`, `n`, `0`, or `keep` → keep;
- any other populated value → delete.

Input:
`output/final_manifest/final_commission_manifest_merged.xlsx`

Output:
`output/final_manifest/filtered_final_manifest.xlsx`

In [ ]:
from pathlib import Path
import re

import pandas as pd
from openpyxl.styles import Font
from openpyxl.utils import get_column_letter

## 1. Paths and input

In [ ]:
def project_root(start=None):
    start = Path(start or Path.cwd()).resolve()
    for directory in (start, *start.parents):
        if (directory / "output" / "final_manifest").is_dir():
            return directory
    raise FileNotFoundError(
        "Could not locate output/final_manifest. "
        "Set ROOT explicitly to the ec_cjeu project path."
    )


ROOT = project_root()

INPUT = (
    ROOT
    / "output"
    / "final_manifest"
    / "final_commission_manifest_merged.xlsx"
)

OUTPUT = (
    ROOT
    / "output"
    / "final_manifest"
    / "filtered_final_manifest.xlsx"
)

if not INPUT.is_file():
    raise FileNotFoundError(
        f"Run notebook 12 first. Missing merged manifest: {INPUT}"
    )

print("Input:", INPUT)
print("Output:", OUTPUT)

## 2. Read the merged manifest

The filtering operates at the merged Commission-decision level.

`candidate_to_delete` remains the manual deletion mechanism. Negative clearances are an explicit exception and are retained.

In [ ]:
book = pd.ExcelFile(INPUT)
sheet = (
    "merged_manifest"
    if "merged_manifest" in book.sheet_names
    else book.sheet_names[0]
)

merged = pd.read_excel(
    INPUT,
    sheet_name=sheet,
    dtype=object,
    keep_default_na=False,
)
merged.columns = [str(column).strip() for column in merged.columns]

if "candidate_to_delete" not in merged.columns:
    raise ValueError(
        "The merged manifest does not contain candidate_to_delete."
    )

print(f"Read {len(merged):,} merged decision rows.")

## 3. Apply the delete flag

Negative clearances are retained.

The safeguard checks the descriptive Commission metadata available in the merged manifest, especially `document_class` and `title`, for the phrase `negative clearance` (including simple hyphen/spacing variants).

This does not create a new substantive classification. It only prevents an explicitly identifiable negative-clearance decision from being removed by the manual delete flag.

In [ ]:
FALSE_MARKERS = {"false", "no", "n", "0", "keep"}


def delete_marked(value):
    if value is None or value is False:
        return False

    if isinstance(value, (int, float)) and not isinstance(value, bool):
        return bool(value) and not pd.isna(value)

    label = str(value).strip()

    return bool(label) and label.casefold() not in FALSE_MARKERS


NEGATIVE_CLEARANCE_RE = re.compile(
    r"\bnegative[\s-]+clearance(?:s)?\b",
    re.IGNORECASE,
)


def is_negative_clearance(row):
    # Restrict the safeguard to descriptive metadata columns available
    # in the merged manifest.
    fields = [
        "document_class",
        "title",
    ]

    text = " | ".join(
        str(row.get(field, "") or "")
        for field in fields
        if field in row.index
    )

    return bool(NEGATIVE_CLEARANCE_RE.search(text))


working = merged.copy()

working["_delete_marked"] = working["candidate_to_delete"].map(
    delete_marked
)
working["_negative_clearance"] = working.apply(
    is_negative_clearance,
    axis=1,
)

# Negative clearances override the manual deletion flag.
working["_delete_final"] = (
    working["_delete_marked"]
    & ~working["_negative_clearance"]
)

filtered = (
    working.loc[~working["_delete_final"]]
    .drop(
        columns=[
            "_delete_marked",
            "_negative_clearance",
            "_delete_final",
        ]
    )
    .reset_index(drop=True)
)

deleted_count = int(working["_delete_final"].sum())
protected_negative_clearances = int(
    (
        working["_delete_marked"]
        & working["_negative_clearance"]
    ).sum()
)

print(f"Deleted rows: {deleted_count:,}")
print(
    "Marked rows retained because they are negative clearances:",
    f"{protected_negative_clearances:,}",
)
print(f"Rows retained: {len(filtered):,}")

## 4. Save the filtered final manifest

In [ ]:
OUTPUT.parent.mkdir(parents=True, exist_ok=True)
pending = OUTPUT.with_name(OUTPUT.stem + ".pending.xlsx")

with pd.ExcelWriter(pending, engine="openpyxl") as writer:
    filtered.to_excel(
        writer,
        sheet_name="filtered_manifest",
        index=False,
    )

    ws = writer.sheets["filtered_manifest"]
    ws.freeze_panes = "C2"
    ws.auto_filter.ref = ws.dimensions

    for cell in ws[1]:
        cell.font = Font(color="FFFFFF", bold=True)
        cell.fill = __import__(
            "openpyxl.styles",
            fromlist=["PatternFill"],
        ).PatternFill("solid", fgColor="19334D")

    for index, column in enumerate(filtered.columns, start=1):
        ws.column_dimensions[get_column_letter(index)].width = min(
            65,
            max(17, len(str(column)) + 3),
        )

pending.replace(OUTPUT)

print("Wrote:", OUTPUT)